# Local Mode Quickstart: End-to-End OMOP CDM Mapping

This notebook demonstrates the complete Portiere SDK pipeline running entirely on your local machine.
No cloud services or API keys are required -- all processing happens locally using the built-in
BM25s knowledge layer and local engines.

## What We Will Do

1. **Install and configure** Portiere in local mode
2. **Create a project** targeting OMOP CDM v5.4
3. **Add a data source** (CSV file)
4. **Map schema** -- automatically map source columns to OMOP CDM tables and columns
5. **Review and approve** schema mappings
6. **Map concepts** -- map source clinical codes (ICD-10, etc.) to standard OMOP concepts
7. **Run ETL** -- generate transformed output files

## Prerequisites

- Python 3.10+
- A CSV data source (we will use a sample `patients.csv`)

## Pipeline Overview

The Portiere SDK follows a 5-stage pipeline:

**Ingest** --> **Profile** --> **Schema Map** --> **ETL Gen** --> **Validate**

In local mode, all stages run on your machine using local models and the BM25s retrieval backend.

## Step 1: Install Portiere

Install the Portiere SDK. The base package includes everything needed for local mode.

In [1]:
!pip install portiere-health


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 1b: Shared Athena Vocabulary Setup

Build (or reuse) BM25s and FAISS indexes from the Athena vocabulary download.
On the first run this takes ~10-30 minutes for FAISS embedding; subsequent runs
reuse the cached indexes.

In [2]:
from pathlib import Path
from portiere.knowledge import build_knowledge_layer

ATHENA_DIR = Path("example_assets/vocabulary_download_v5")
VOCAB_DIR  = ATHENA_DIR / "indexes"
BM25S_CORPUS = VOCAB_DIR / "concepts.json"
FAISS_INDEX  = VOCAB_DIR / "concepts.index"
FAISS_META   = VOCAB_DIR / "concepts.meta.json"

# Build indexes from Athena on first run (~10-30 min for FAISS embedding)
if not BM25S_CORPUS.exists():
    paths = build_knowledge_layer(
        athena_path=ATHENA_DIR,
        output_path=VOCAB_DIR,
        backend="hybrid",
        vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    )
    print(f"Built indexes: {paths}")
else:
    print(f"Using existing indexes at {VOCAB_DIR}")

Using existing indexes at example_assets/vocabulary_download_v5/indexes


## Step 2: Import and Configure

We configure Portiere with a **local knowledge layer**. Portiere automatically infers that
both storage and AI inference should run locally when you provide a knowledge layer config
without an API key:

- `knowledge_layer` with `backend="bm25s"` -- uses BM25s for terminology retrieval (no external services needed)
- No `api_key` set -- storage and pipeline default to local

There is no need to explicitly set `mode` or `pipeline` -- Portiere infers the right
behavior from your configuration.

In [3]:
import portiere
from portiere import PortiereConfig, KnowledgeLayerConfig

config = PortiereConfig(
    knowledge_layer=KnowledgeLayerConfig(
        backend="bm25s",
        bm25s_corpus_path=str(BM25S_CORPUS),
    ),
)

print(f"Mode: {config.effective_mode}")
print(f"Pipeline: {config.effective_pipeline}")
print(f"Knowledge backend: {config.knowledge_layer.backend}")

Mode: local
Pipeline: local
Knowledge backend: bm25s


## Step 3: Create a Project

Initialize a new mapping project. The project defines:

- **name** -- a human-readable identifier for this migration
- **task** -- the project's purpose: `"standardize"` (default) maps raw data to a target standard
- **target_model** -- the target CDM (here, OMOP CDM v5.4)
- **vocabularies** -- which standard vocabularies to use for concept mapping

The `portiere.init()` call sets up the local project directory and loads the target schema.

In [4]:
from portiere.engines import PolarsEngine

project = portiere.init(
    name="Hospital OMOP Migration",
    engine=PolarsEngine(),
    target_model="omop_cdm_v5.4",
    vocabularies=["SNOMED", "LOINC", "RxNorm", "ICD10CM"],
    config=config,
)

print(project)

2026-04-16 00:25:07 [info     ] PolarsEngine initialized      


2026-04-16 00:25:07 [debug    ] local_storage.initialized      base_dir=/Users/kuangsmacbook/.portiere/projects


Project(name='Hospital OMOP Migration', task='standardize', target_model='omop_cdm_v5.4', mode='local')


## Step 4: Add a Data Source

Add a CSV file as a data source. The SDK will read the file, detect its format, and prepare it
for schema profiling and mapping.

The returned dictionary contains metadata about the ingested source, including its name, format,
detected columns, and row count.

In [5]:
source = project.add_source("example_assets/01_local_mode_quickstart/patients.csv")

print(f"Source added: {source['name']}, format: {source['format']}")
print(f"Columns detected: {source.get('columns', 'N/A')}")
print(f"Row count: {source.get('row_count', 'N/A')}")

2026-04-16 00:25:07 [info     ] project.source_added           project='Hospital OMOP Migration' source=patients


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

2026-04-16 00:25:10 [info     ] gx_profiler.profiled           columns=11 rows=15 source=patients


2026-04-16 00:25:10 [info     ] project.profiled               source=patients


Source added: patients, format: csv
Columns detected: ['patient_id', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'ethnicity', 'address', 'city', 'state', 'zip_code']
Row count: 15


## Step 5: Map Schema

Schema mapping automatically matches source columns to OMOP CDM target tables and columns.
The SDK uses the knowledge layer to find the best matches and assigns confidence scores.

Confidence routing determines the status of each mapping:

| Confidence Range | Status | Action |
|-----------------|--------|--------|
| >= 0.95 | AUTO_ACCEPTED | No review needed |
| 0.80 - 0.95 | VERIFY | Quick verification recommended |
| 0.70 - 0.80 | REVIEW | Manual review needed |
| < 0.70 | MANUAL | Requires manual mapping |

In [6]:
schema_map = project.map_schema(source)

print(schema_map)
print(f"\nSummary: {schema_map.summary()}")

2026-04-16 00:25:10 [info     ] Stage 2: Schema mapping (local) embedding_model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext embedding_provider=huggingface target_model=omop_cdm_v5.4


2026-04-16 00:25:10 [info     ] local_schema_mapper.loading_model model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface target_standard=omop_cdm_v5.4


2026-04-16 00:25:10 [info     ] embedding_gateway.initialized  model=cambridgeltl/SapBERT-from-PubMedBERT-fulltext provider=huggingface


2026-04-16 00:25:10 [info     ] local_schema_mapper.computing_target_embeddings standard=omop_cdm_v5.4 targets=67


2026-04-16 00:25:12 [warning  ] local_schema_mapper.init_failed: sentence-transformers is required for HuggingFace embeddings. Install with: pip install sentence-transformers


2026-04-16 00:25:12 [info     ] Stage 2 complete               auto=10 review=0 total=11


2026-04-16 00:25:12 [info     ] local_storage.schema_mapping_saved items_count=11 project='Hospital OMOP Migration'


2026-04-16 00:25:12 [info     ] project.schema_mapped          auto=10 project='Hospital OMOP Migration' total=11


items=[SchemaMappingItem(source_column='patient_id', source_table='', target_table='person', target_column='person_id', confidence=0.95, status=<MappingStatus.AUTO_ACCEPTED: 'auto_accepted'>, candidates=[], override_target_table=None, override_target_column=None), SchemaMappingItem(source_column='first_name', source_table='', target_table='person', target_column='person_source_value', confidence=0.95, status=<MappingStatus.AUTO_ACCEPTED: 'auto_accepted'>, candidates=[], override_target_table=None, override_target_column=None), SchemaMappingItem(source_column='last_name', source_table='', target_table='person', target_column='person_source_value', confidence=0.5, status=<MappingStatus.UNMAPPED: 'unmapped'>, candidates=[], override_target_table=None, override_target_column=None), SchemaMappingItem(source_column='date_of_birth', source_table='', target_table='person', target_column='birth_datetime', confidence=0.95, status=<MappingStatus.AUTO_ACCEPTED: 'auto_accepted'>, candidates=[], ove

## Step 6: Review Schema Mappings

Let's inspect the individual mapping items. Each item shows:

- **source_column** -- the original column name from your CSV
- **target_table** / **target_column** -- the mapped OMOP CDM destination
- **confidence** -- how confident the algorithm is in this mapping (0.0 to 1.0)
- **status** -- the current mapping status (AUTO_ACCEPTED, VERIFY, REVIEW, MANUAL, UNMAPPED)

In [7]:
print("Schema Mapping Results (first 5 items):")
print("-" * 80)

for item in schema_map.items[:5]:
    print(
        f"  {item.source_column} -> {item.target_table}.{item.target_column} "
        f"(confidence: {item.confidence:.2f}, status: {item.status.value})"
    )

Schema Mapping Results (first 5 items):
--------------------------------------------------------------------------------
  patient_id -> person.person_id (confidence: 0.95, status: auto_accepted)
  first_name -> person.person_source_value (confidence: 0.95, status: auto_accepted)
  last_name -> person.person_source_value (confidence: 0.50, status: unmapped)
  date_of_birth -> person.birth_datetime (confidence: 0.95, status: auto_accepted)
  gender -> person.gender_concept_id (confidence: 0.95, status: auto_accepted)


### Check Items That Need Review

The `needs_review()` method returns only the items that require human attention
(those with status VERIFY, REVIEW, or MANUAL).

In [8]:
review_items = schema_map.needs_review()

print(f"Items needing review: {len(review_items)}")
for item in review_items:
    print(
        f"  [{item.status.value}] {item.source_column} -> "
        f"{item.target_table}.{item.target_column} (confidence: {item.confidence:.2f})"
    )

Items needing review: 0


## Step 7: Approve Schema Mappings

After reviewing, approve all mappings and finalize the schema mapping.

- `approve_all()` -- marks all remaining items as approved
- `finalize()` -- locks the schema mapping so it can be used in ETL generation

In a production workflow, you would selectively approve or override individual items
before calling `finalize()`.

In [9]:
schema_map.approve_all()
schema_map.finalize()

print("Schema mapping finalized.")
print(f"Updated summary: {schema_map.summary()}")

Schema mapping finalized.
Updated summary: {'total': 11, 'auto_accepted': 10, 'needs_review': 0, 'approved': 0, 'unmapped': 1, 'auto_rate': 90.9090909090909}


## Step 8: Map Concepts

Concept mapping translates source clinical codes (ICD-10-CM, CPT, NDC, etc.) to standard
OMOP vocabulary concepts (SNOMED, LOINC, RxNorm, etc.).

There are two approaches:

### Option A: Auto-discover codes from the source

When you pass a `source` to `map_concepts()`, Portiere auto-detects columns likely to
contain clinical codes (columns ending in `_code` like `diagnosis_code`, `drug_code`,
`test_code`, etc.) and maps their values automatically.

Since our `patients.csv` contains only demographic data (no clinical code columns),
auto-discovery will correctly find 0 codes. In a real pipeline, you would use this
with a `diagnoses.csv` or `medications.csv` that has code columns.

In [10]:
concept_map = project.map_concepts(source=source)
print(f"Auto-discovery result: {concept_map.summary()}")
print("(No clinical code columns found in patients.csv -- this is expected)\n")

2026-04-16 00:25:12 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-04-16 00:25:12 [info     ] Stage 3 complete               auto_rate=0.0% total_codes=0


2026-04-16 00:25:12 [info     ] local_storage.concept_mapping_saved items_count=0 project='Hospital OMOP Migration'


2026-04-16 00:25:12 [info     ] project.concepts_mapped        auto_rate=0 project='Hospital OMOP Migration' total=0


Auto-discovery result: {'total': 0, 'auto_mapped': 0, 'needs_review': 0, 'manual_required': 0, 'auto_rate': 0, 'coverage': 0}
(No clinical code columns found in patients.csv -- this is expected)



### Option B: Map explicit codes (used below)

If you already know which codes you need to map, pass them directly. The knowledge layer
searches across all configured vocabularies to find the best standard concept for each code.

In [11]:
concept_map = project.map_concepts(
    codes=["E11.9", "I10", "J18.9", "N18.6", "Z87.891"]
)

print(concept_map)
print(f"\nSummary: {concept_map.summary()}")

2026-04-16 00:25:12 [info     ] Stage 3: Concept mapping (local) columns=[] knowledge_backend=bm25s


2026-04-16 00:25:12 [info     ] knowledge.creating_backend     backend=bm25s path=example_assets/vocabulary_download_v5/indexes/concepts.json


Split strings:   0%|          | 0/623910 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/623910 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-16 00:25:22 [info     ] Stage 3 complete               auto_rate=20.0% total_codes=5


2026-04-16 00:25:22 [info     ] local_storage.concept_mapping_saved items_count=5 project='Hospital OMOP Migration'


2026-04-16 00:25:22 [info     ] project.concepts_mapped        auto_rate=20.0 project='Hospital OMOP Migration' total=5


items=[ConceptMappingItem(source_code='E11.9', source_description='E11.9', source_column=None, source_count=1, target_concept_id=None, target_concept_name='', target_vocabulary_id='', target_domain_id='', confidence=0.0, method=<ConceptMappingMethod.MANUAL: 'manual'>, candidates=[], provenance={}), ConceptMappingItem(source_code='I10', source_description='I10', source_column=None, source_count=1, target_concept_id=None, target_concept_name='', target_vocabulary_id='', target_domain_id='', confidence=0.0, method=<ConceptMappingMethod.MANUAL: 'manual'>, candidates=[], provenance={}), ConceptMappingItem(source_code='J18.9', source_description='J18.9', source_column=None, source_count=1, target_concept_id=None, target_concept_name='', target_vocabulary_id='', target_domain_id='', confidence=0.0, method=<ConceptMappingMethod.MANUAL: 'manual'>, candidates=[], provenance={}), ConceptMappingItem(source_code='N18.6', source_description='N18.6', source_column=None, source_count=1, target_concept

### Review Concept Mapping Results

Inspect each concept mapping result. Each item includes:

- **source_code** -- the original clinical code
- **target_concept_name** -- the matched standard concept name
- **method** -- how the mapping was determined (AUTO, VERIFY, MANUAL, OVERRIDE)
- **confidence** -- confidence score for the mapping

In [12]:
print("Concept Mapping Results:")
print("-" * 80)

for item in concept_map.items:
    print(
        f"  {item.source_code}: {item.target_concept_name} "
        f"({item.method.value}, confidence: {item.confidence:.2f})"
    )

Concept Mapping Results:
--------------------------------------------------------------------------------
  E11.9:  (manual, confidence: 0.00)
  I10:  (manual, confidence: 0.00)
  J18.9:  (manual, confidence: 0.00)
  N18.6: N18 (auto, confidence: 8.08)
  Z87.891:  (manual, confidence: 0.00)


### Check Concept Mappings Needing Review

In [13]:
review_concepts = concept_map.needs_review()

print(f"Concept mappings needing review: {len(review_concepts)}")
for item in review_concepts:
    print(f"  {item.source_code}: {item.target_concept_name} (confidence: {item.confidence:.2f})")

Concept mappings needing review: 0


## Step 10: Run ETL

Generate the transformed output. The ETL step takes the finalized schema and concept mappings
and produces OMOP CDM-compliant output files.

Parameters:

- **source** -- the data source to transform
- **output_dir** -- where to write the output files
- **schema_mapping** -- the finalized schema mapping
- **concept_mapping** -- the concept mapping to apply

In [14]:
etl_result = project.run_etl(
    source,
    output_dir="./output",
    schema_mapping=schema_map,
    concept_mapping=concept_map,
)

print(f"ETL complete. Output: {etl_result}")

2026-04-16 00:25:22 [info     ] Reading source                 format=csv path=example_assets/01_local_mode_quickstart/patients.csv


2026-04-16 00:25:22 [info     ] Processing table               columns=6 table=person


2026-04-16 00:25:22 [info     ] Processing table               columns=4 table=location


2026-04-16 00:25:22 [info     ] ETL complete                   duration=0.005336 success=True


2026-04-16 00:25:22 [info     ] project.etl_complete           output_dir=./output project='Hospital OMOP Migration'


ETL complete. Output: success=True started_at=datetime.datetime(2026, 4, 15, 17, 25, 22, 270270, tzinfo=datetime.timezone.utc) completed_at=datetime.datetime(2026, 4, 15, 17, 25, 22, 275606, tzinfo=datetime.timezone.utc) duration_seconds=0.005336 source_path='example_assets/01_local_mode_quickstart/patients.csv' source_rows_read=15 output_path='./output' tables=[TableResult(table_name='person', rows_written=15, columns=['person_id', 'person_source_value', 'birth_datetime', 'gender_concept_id', 'race_concept_id', 'ethnicity_concept_id'], output_path='./output/person.csv', concept_columns_mapped=[], unmapped_concept_count=0), TableResult(table_name='location', rows_written=15, columns=['address_1', 'city', 'state', 'zip'], output_path='./output/location.csv', concept_columns_mapped=[], unmapped_concept_count=0)] total_rows_written=30 schema_mappings_applied=10 concept_mappings_applied=0 unmapped_columns=['last_name'] engine_name='polars' errors=[] warnings=[]


## Step 11: Validate Output (Optional)

Run validation checks on the ETL output to ensure data quality and target model conformance.

In [15]:
validation = project.validate(etl_result=etl_result)

print(f"Validation result: {validation}")

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

2026-04-16 00:25:22 [info     ] gx_validator.validated         completeness=0.44 conformance=1.00 passed=False plausibility=0.44 table=location


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-04-16 00:25:22 [info     ] gx_validator.validated         completeness=0.67 conformance=1.00 passed=False plausibility=0.70 table=person


2026-04-16 00:25:22 [info     ] project.validated              all_passed=False project='Hospital OMOP Migration' tables=2


Validation result: {'tables': [{'table_name': 'location', 'passed': False, 'completeness_score': 0.4444444444444444, 'conformance_score': 1.0, 'plausibility_score': 0.4444444444444444, 'gx_result': {'success': False, 'results': [{'success': False, 'expectation_config': {'type': 'expect_column_to_exist', 'kwargs': {'batch_id': 'validate_location-location', 'column': 'location_id'}, 'meta': {}, 'id': 'b239dfcc-101e-4f30-bc95-76314c9433c7', 'severity': 'critical'}, 'result': {}, 'meta': {}, 'exception_info': {'raised_exception': False, 'exception_traceback': None, 'exception_message': None}}, {'success': True, 'expectation_config': {'type': 'expect_column_to_exist', 'kwargs': {'batch_id': 'validate_location-location', 'column': 'address_1'}, 'meta': {}, 'id': '0abbd20b-b0d3-4db4-90a8-a8c53949492a', 'severity': 'critical'}, 'result': {}, 'meta': {}, 'exception_info': {'raised_exception': False, 'exception_traceback': None, 'exception_message': None}}, {'success': False, 'expectation_config

In [16]:
validation

{'tables': [{'table_name': 'location',
   'passed': False,
   'completeness_score': 0.4444444444444444,
   'conformance_score': 1.0,
   'plausibility_score': 0.4444444444444444,
   'gx_result': {'success': False,
    'results': [{'success': False,
      'expectation_config': {'type': 'expect_column_to_exist',
       'kwargs': {'batch_id': 'validate_location-location',
        'column': 'location_id'},
       'meta': {},
       'id': 'b239dfcc-101e-4f30-bc95-76314c9433c7',
       'severity': 'critical'},
      'result': {},
      'meta': {},
      'exception_info': {'raised_exception': False,
       'exception_traceback': None,
       'exception_message': None}},
     {'success': True,
      'expectation_config': {'type': 'expect_column_to_exist',
       'kwargs': {'batch_id': 'validate_location-location',
        'column': 'address_1'},
       'meta': {},
       'id': '0abbd20b-b0d3-4db4-90a8-a8c53949492a',
       'severity': 'critical'},
      'result': {},
      'meta': {},
      'ex

---

## Summary

In this notebook we completed a full end-to-end OMOP CDM mapping workflow using Portiere
in **local mode**:

1. Configured the SDK for fully local operation (no cloud dependencies)
2. Created a project targeting OMOP CDM v5.4 with standard vocabularies
3. Added a CSV data source and mapped its schema to OMOP CDM tables
4. Reviewed, approved, and finalized schema mappings
5. Mapped clinical codes to standard OMOP concepts
6. Generated ETL output and ran validation

## Next Steps

- **Sync to cloud**: See [Bonus: Sync to Cloud](#bonus-sync-to-cloud) below to push your
  local work to Portiere Cloud for team review.
- **Cloud pipeline**: See `02_cloud_pipeline.ipynb` to use Portiere's cloud AI inference
  while keeping your data local.
- **BYO-LLM**: See `03_byo_llm_providers.ipynb` to use your own LLM provider (OpenAI,
  Anthropic, Ollama, Azure, or AWS Bedrock).
- **Data profiling**: Install `pip install portiere-health[quality]` and use `project.profile(source)`
  to get detailed data quality reports before mapping.
- **Selective review**: Instead of `approve_all()`, use individual `item.approve()` and
  `item.override()` calls for fine-grained control over mappings.

---

## Bonus: Sync to Cloud

Finished your local work and want to share it with your team? Use `push()` to upload
your project to Portiere Cloud. Team members can review, approve, and override mappings
in the web UI. When they are done, `pull()` brings the changes back to your local machine.

This is the **recommended workflow** for teams that want local development speed plus
cloud collaboration.

### Requirements

- A Portiere Cloud API key (`pt_sk_...`)
- The project was created with `api_key` configured (or added later)

In [17]:
# Push local project to Portiere Cloud (cloud feature — skipped in open-source SDK)
try:
    cloud_id = project.push()
    print(f"Project synced to cloud: {cloud_id}")
except NotImplementedError as e:
    print(f"Cloud sync not available in open-source SDK: {e}")
    print("For cloud storage and collaboration, see https://portiere.io")


Cloud sync not available in open-source SDK: Cloud sync is not available in the open-source SDK. For cloud storage, sync, and collaboration, see https://portiere.io
For cloud storage and collaboration, see https://portiere.io


The `push()` / `pull()` pattern lets you iterate locally at full speed, then sync to cloud
only when you are ready for team review. See `02_cloud_pipeline.ipynb` for fully cloud-managed
projects where all artifacts are stored and tracked on Portiere Cloud automatically.